# Lab 10-03 — Evidence verification loop (SciFact claims)

**Track 10 · Agentic RAG** — generation metrics grade answers; attribution metrics grade *whether the answer is actually backed by the retrieved evidence*. This lab implements the evidence-verification half of that idea inline over the SciFact corpus.

This notebook is **self-contained**: it imports `langchain-ollama` and `langchain-huggingface` directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder (`HuggingFaceEmbeddings`), the cosine rank over the whole corpus, and the three-way verdict loop (`SUPPORTED` / `REFUTED` / `NOT_ENOUGH_INFO`) all appear as plain code in the cells below. LangChain has no turnkey claim-verification flow, so the verifier is hand-rolled inline — the same gap the shared `src/tools/verifier.py` fills for the Curriculum series.

```text
SciFact claim
  -> BGE embeddings over the whole 5183-doc corpus (once, cached)
  -> cosine rank -> top-5 candidate evidence passages
  -> ChatOllama (json_object) three-way verdict:
       SUPPORTED | REFUTED | NOT_ENOUGH_INFO
  -> verdict + reason + evidence titles per claim
  -> verification gate (structural, tolerant)
```

For each claim we (1) embed the claim and rank the whole 5183-doc corpus by cosine similarity to retrieve top-5 candidate evidence passages, then (2) ask the local LLM — via `json_object` — for a three-way verdict:

* `SUPPORTED`       — the evidence backs the claim,
* `REFUTED`         — the evidence contradicts the claim,
* `NOT_ENOUGH_INFO` — the evidence neither supports nor refutes it.

The corpus is embedded exactly ONCE (CPU, ~5-6 minutes for 5183 docs) and cached in the experiment dict; per-claim retrieval is then a pure cosine rank over the cached vectors. The gate is structural and tolerant: it does NOT require agreement with SciFact's gold labels — a local 7B model verifying 4 claims is not a benchmark run. It checks that every claim got a valid verdict, a non-empty reason (or the explicit "parse failure" fallback), and that the loop terminated.


## Setup

Two prerequisites must hold before this notebook will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM behind the verdict calls (wired up inline with `langchain-ollama`'s `ChatOllama`, `base_url="http://localhost:11434"`). Fully local: no API key, no quota.
- **SciFact on disk** — `Data/corpus/scifact/data/corpus.jsonl` (5183 docs) + `Data/corpus/scifact/data/claims_dev.jsonl`, already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-ollama` and `langchain-huggingface`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   langchain-ollama      -> ChatOllama, the local LLM backend (qwen2.5-coder:7b)
#   langchain-huggingface -> HuggingFaceEmbeddings, the local BGE embedder
%pip install -q langchain-ollama langchain-huggingface


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

# LangChain — the only libraries this notebook needs. Nothing is imported
# from the repo's src/ component library.
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_ollama import ChatOllama  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_CLAIMS = 4` bounds the demo to four dev claims (each = 1 LLM verdict call, ~5-8 LLM calls total); `TOP_K = 5` is how many candidate evidence passages each claim retrieves; BGE runs on CPU because Ollama holds most of the GPU VRAM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
CORPUS_PATH = Path("Data/corpus/scifact/data/corpus.jsonl")
CLAIMS_PATH = Path("Data/corpus/scifact/data/claims_dev.jsonl")
N_CLAIMS = 4  # each claim = 1 LLM verdict call; ~5-8 LLM calls total
TOP_K = 5  # candidate evidence passages retrieved per claim
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — claims (deterministic head) and the full SciFact corpus

`load_corpus` reads every corpus record as `{"doc_id", "title", "abstract"}`; `corpus_texts` flattens each doc to one searchable string — `title. <abstract sentences>` — which is what gets embedded; `load_claims` takes the first `n` claims from the dev set (each carries its `id`).


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — claims (deterministic head) and the full SciFact corpus
# --------------------------------------------------------------------------
def load_corpus(path: Path) -> list[dict]:
    """All corpus records: ``{"doc_id", "title", "abstract"}``."""
    docs: list[dict] = []
    with open(path) as f:
        for line in f:
            docs.append(json.loads(line))
    return docs


def corpus_texts(docs: list[dict]) -> list[str]:
    """One searchable string per doc: ``title. <abstract sentences>``."""
    return [f"{doc['title']}. {' '.join(doc['abstract'])}" for doc in docs]


def load_claims(path: Path, n: int) -> list[dict]:
    """First ``n`` claims from the dev set (each carries its ``id``)."""
    claims: list[dict] = []
    with open(path) as f:
        for line in f:
            claims.append(json.loads(line))
            if len(claims) >= n:
                break
    return claims


## 3. Experiment — embed the corpus once, then verify each claim

Everything is built inline. `_OllamaLLM` is a hand-rolled stand-in for `src/llms/ollama.py` (ChatOllama + `invoke` / `json_object` with code-fence stripping and retries); `_bge_embedder` is the hand-rolled stand-in for `src/embeddings/bge.py` (`HuggingFaceEmbeddings`, `normalize_embeddings=True`).

The verifier is hand-rolled inline too — LangChain has no turnkey claim-verification flow, which is exactly what `src/tools/verifier.py` supplies for the Curriculum series, and this notebook reproduces that logic verbatim: `VALID_VERDICTS` is the three-value verdict set; `retrieve_evidence` ranks the corpus by cosine against the claim (reusing `corpus_embeddings` when the caller precomputed them); `verify_claim` asks the LLM for a JSON verdict and NEVER crashes — a bad/absent JSON falls back to `{"verdict": "NOT_ENOUGH_INFO", "reason": "parse failure"}`; `verify_loop` chains retrieve → verify over every claim with an optional progress callback.

`run_experiment` embeds the 5183-doc corpus exactly ONCE (~5-6 min on CPU), keeps the vectors in the experiment dict, and runs the loop over the first `N_CLAIMS` dev claims.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed the corpus once, then verify each claim
# --------------------------------------------------------------------------
class _OllamaLLM:
    """Inline stand-in for src/llms/ollama.py: ChatOllama + invoke/json_object.

    Wraps langchain-ollama's ChatOllama (local qwen2.5-coder:7b, fully local)
    and exposes the same contract the lab's components rely on: ``invoke``
    returns plain text, ``json_object`` strips code fences and retries JSON.
    """

    def __init__(self, model: str = "qwen2.5-coder:7b", temperature: float = 0.0,
                 base_url: str = "http://localhost:11434"):
        self.model = model
        self.temperature = temperature
        self._llm = ChatOllama(model=model, temperature=temperature, base_url=base_url)

    def invoke(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content

    @staticmethod
    def _strip_code_fence(text: str) -> str:
        """Remove a surrounding markdown code fence (```json ... ```)."""
        lines = text.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        return "\n".join(lines).strip()

    def json_object(self, prompt: str, retries: int = 2) -> dict:
        """Ask the model to output ONLY a JSON object and parse it."""
        text = ""
        for attempt in range(retries + 1):
            full = prompt if attempt == 0 else prompt + (
                "\n\nRespond with ONLY valid JSON, no markdown.")
            text = self.invoke(full)
            try:
                parsed = json.loads(self._strip_code_fence(text))
                if isinstance(parsed, (dict, list)):
                    return parsed
            except (json.JSONDecodeError, ValueError):
                pass
        return {"error": f"could not parse JSON after {retries + 1} attempts",
                "raw": text}


def _bge_embedder() -> HuggingFaceEmbeddings:
    """Inline stand-in for src/embeddings/bge.py: BGE via HuggingFaceEmbeddings.

    bge models require normalized embeddings for cosine similarity, so
    ``normalize_embeddings=True`` is set exactly like the shared component.
    """
    return HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # bge needs cosine-normalized vectors
    )


# -- hand-rolled verifier (inline stand-in for src/tools/verifier.py) -------

#: The only verdicts the pipeline knows how to report.
VALID_VERDICTS = frozenset({"SUPPORTED", "REFUTED", "NOT_ENOUGH_INFO"})


def _cosine(a: list[float], b: list[float]) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def retrieve_evidence(
    question_or_claim: str,
    corpus_texts: list[str],
    embedder,
    top_k: int = 5,
    corpus_embeddings: list[list[float]] | None = None,
) -> list[dict]:
    """Rank ``corpus_texts`` against the claim and return the top ``top_k``.

    ``corpus_embeddings`` (optional) lets a caller embed a large corpus once
    and reuse the vectors across many claims — the whole point for the
    5183-doc SciFact corpus, where per-claim re-embedding is wasteful.
    """
    query_vec = embedder.embed_documents([question_or_claim])[0]
    if corpus_embeddings is None:
        corpus_embeddings = embedder.embed_documents(list(corpus_texts))
    scored = [
        (_cosine(query_vec, vec), index)
        for index, vec in enumerate(corpus_embeddings)
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [
        {
            "text": corpus_texts[index],
            "index": index,
            "score": round(score, 4),
        }
        for score, index in scored[:top_k]
    ]


def verify_claim(llm, claim: str, evidence_texts: list[str]) -> dict:
    """Ask the LLM for a three-way verdict on ``claim`` against the evidence.

    Never crashes: a bad/absent JSON verdict falls back to
    ``{"verdict": "NOT_ENOUGH_INFO", "reason": "parse failure"}``.
    """
    evidence_block = "\n\n---\n\n".join(
        f"[{i + 1}] {text}" for i, text in enumerate(evidence_texts)
    )
    prompt = (
        "You are an evidence verifier for scientific claims.\n"
        "Decide whether the evidence passages support, refute, or say "
        "nothing about the claim.\n"
        "Rules:\n"
        '- Verdict must be EXACTLY one of: "SUPPORTED", "REFUTED", '
        '"NOT_ENOUGH_INFO".\n'
        '- "SUPPORTED": the evidence explicitly backs the claim.\n'
        '- "REFUTED": the evidence explicitly contradicts the claim.\n'
        '- "NOT_ENOUGH_INFO": the evidence neither supports nor refutes '
        "the claim (including when it is irrelevant).\n"
        '- Output ONLY JSON: {"verdict": "SUPPORTED", '
        '"reason": "one-sentence justification"}\n'
        "\n"
        f"Claim:\n{claim}\n\n"
        f"Evidence:\n{evidence_block}"
    )
    result = llm.json_object(prompt)
    if isinstance(result, dict) and "error" not in result:
        verdict = str(result.get("verdict", "")).strip().upper()
        if verdict in VALID_VERDICTS:
            reason = str(result.get("reason", "")).strip()
            return {"verdict": verdict, "reason": reason or "no reason given"}
    return {"verdict": "NOT_ENOUGH_INFO", "reason": "parse failure"}


def verify_loop(
    llm,
    embedder,
    claims: list[str],
    corpus_texts: list[str],
    top_k: int = 5,
    progress=None,
    corpus_embeddings: list[list[float]] | None = None,
) -> list[dict]:
    """Verify every claim: retrieve top-k evidence, then ask the LLM.

    The corpus is embedded exactly once (either precomputed via
    ``corpus_embeddings`` or here on first use) and reused for all claims.
    Returns one dict per claim:
    ``{"claim", "verdict", "reason", "evidence_indices"}``.
    """
    if corpus_embeddings is None:
        corpus_embeddings = embedder.embed_documents(list(corpus_texts))
    results: list[dict] = []
    total = len(claims)
    for done, claim in enumerate(claims, start=1):
        hits = retrieve_evidence(
            claim,
            corpus_texts,
            embedder,
            top_k=top_k,
            corpus_embeddings=corpus_embeddings,
        )
        verdict = verify_claim(llm, claim, [hit["text"] for hit in hits])
        results.append(
            {
                "claim": claim,
                "verdict": verdict["verdict"],
                "reason": verdict["reason"],
                "evidence_indices": [hit["index"] for hit in hits],
            }
        )
        if progress is not None:
            progress(done, total)
    return results


def run_experiment() -> dict:
    corpus = load_corpus(CORPUS_PATH)
    texts = corpus_texts(corpus)
    claims = load_claims(CLAIMS_PATH, N_CLAIMS)

    llm = _OllamaLLM()  # local qwen2.5-coder:7b via ChatOllama; json_object returns the verdict
    embedder = _bge_embedder()  # inline BGE via HuggingFaceEmbeddings

    # Embed the 5183-doc corpus ONCE (~5-6 min on CPU) and keep the vectors in
    # the experiment dict so every claim reuses them.
    t0 = time.perf_counter()
    corpus_embeddings = embedder.embed_documents(texts)
    embed_s = time.perf_counter() - t0

    results = verify_loop(
        llm,
        embedder,
        [claim["claim"] for claim in claims],
        texts,
        top_k=TOP_K,
        progress=lambda done, total: print(
            f"  verified {done}/{total} claims", flush=True
        ),
        corpus_embeddings=corpus_embeddings,
    )
    total_s = time.perf_counter() - t0

    rows = []
    for claim, result in zip(claims, results):
        rows.append(
            {
                "id": claim["id"],
                "claim": result["claim"],
                "verdict": result["verdict"],
                "reason": result["reason"],
                "evidence_indices": result["evidence_indices"],
                "evidence_titles": [
                    corpus[i]["title"] for i in result["evidence_indices"]
                ],
            }
        )
    return {
        "rows": rows,
        "corpus_size": len(corpus),
        "embed_s": embed_s,
        "total_s": total_s,
        "corpus_embeddings": corpus_embeddings,  # cached for reuse/inspection
        "distribution": dict(Counter(row["verdict"] for row in rows)),
    }


## 4. Demo — print the artifact

The demo prints the artifact per claim: the claim text, up to three retrieved evidence titles, the three-way verdict and its reason; then the verdict distribution over the claims and the takeaway: verification is RAG's answer-quality gate — retrieve the most plausible evidence, then ask the model whether it actually supports, refutes, or ignores the claim. The verdict is a machine-checkable signal (not free text), and the "parse failure" fallback guarantees the pipeline never crashes on a stubborn local model. Agreement with SciFact gold labels is intentionally not enforced in the gate.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 10-03 — Evidence verification loop (SciFact claims)")
    print(f"{len(exp['rows'])} claims over {exp['corpus_size']} docs; "
          f"embed {exp['embed_s']:.0f}s, total {exp['total_s']:.0f}s")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nC{i} ({row['id']}): {row['claim'][:100]}")
        for title in row["evidence_titles"][:3]:
            print(f"    evidence: {title[:95]}")
        print(f"    verdict : {row['verdict']}")
        print(f"    reason  : {row['reason'][:140]}")

    print(f"\n[4] Verdict distribution over {len(exp['rows'])} claims")
    for verdict in ("SUPPORTED", "REFUTED", "NOT_ENOUGH_INFO"):
        print(f"    {verdict:<16} {exp['distribution'].get(verdict, 0)}")
    print(f"    {'TOTAL':<16} {len(exp['rows'])}")

    print(f"\n[5] Takeaway")
    print("    Verification is RAG's answer-quality gate: retrieve the most")
    print("    plausible evidence, then ask the model whether it actually")
    print("    supports, refutes, or ignores the claim. The verdict is a")
    print("    machine-checkable signal (not free text), and the 'parse")
    print("    failure' fallback guarantees the pipeline never crashes on a")
    print("    stubborn local model. Agreement with SciFact gold labels is")
    print("    intentionally not enforced in the gate.")


## 5. Verification gate

The gate is structural and tolerant — it does NOT require agreement with SciFact's gold labels. It checks: all `N_CLAIMS` claims were verified (loop terminated), every verdict is one of `SUPPORTED`/`REFUTED`/`NOT_ENOUGH_INFO`, every verdict has a non-empty reason (or a parse-failure note), and every claim retrieved exactly `TOP_K` evidence passages.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    rows = exp["rows"]

    checks.append((f"all {N_CLAIMS} claims were verified (loop terminated)",
                   len(rows) == N_CLAIMS))
    checks.append(("every verdict is one of SUPPORTED/REFUTED/NOT_ENOUGH_INFO",
                   all(row["verdict"] in VALID_VERDICTS for row in rows)))
    checks.append(("every verdict has a non-empty reason (or a parse-failure note)",
                   all(row["reason"].strip() for row in rows)))
    checks.append((f"every claim retrieved {TOP_K} evidence passages",
                   all(len(row["evidence_indices"]) == TOP_K for row in rows)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

One ~5-6 minute corpus embedding pass, then 4 verdict LLM calls (~5-8 calls total) — expect around 10 minutes end to end. No downloads, no API calls. `exp` holds everything the demo and gate need (including the cached corpus embeddings).


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per claim: evidence titles, three-way verdict, and reason — then the verdict distribution and takeaway.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that the SciFact corpus/claims files are intact.


In [ ]:
verify_gate(exp)
